# Error redistribution in the current deduplicated experiments

Use this project's results for 360 AusGrid2013 nodes and 10 GEF2017 nodes.
AusGrid2013 covers a 1-step horizon, and GEF2017 covers 1/3/6-step horizons. Both datasets include FHF, HFF, and HFFPS.
Compare each reconciliation method only with the base from the same training approach. Obtain scales from the corresponding saved training targets,
average within the current semantic levels, and use a calendar-day block bootstrap for 95% intervals.

In [ ]:
# Locate the project package from a notebook folder or the project root.
from pathlib import Path
import os
import sys
_pphf_start = Path.cwd().resolve()
_pphf_override = os.environ.get("PPHF_PROJECT_ROOT") or os.environ.get("DP_FHF_PROJECT_ROOT")
_pphf_candidates = ([Path(_pphf_override).expanduser()] if _pphf_override else []) + [_pphf_start, *_pphf_start.parents]
PROJECT_ROOT = next((p.resolve() for p in _pphf_candidates if (p / "PPHF_modules" / "fhf" / "FHF_runner.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot locate PPHF_modules. Set PPHF_PROJECT_ROOT or start Jupyter inside the project.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ROOT = PROJECT_ROOT
import json
import pandas as pd
from IPython.display import Image, display
from PPHF_modules.results_check.error_analysis.error_redistribution_analysis import run_all

In [ ]:
OUTPUT_DIR = ROOT / 'Results_check' / 'deduplication_results' / 'error_analysis'
N_BOOTSTRAP = 2000
RECOMPUTE = False
manifest = run_all(root=ROOT, output_root=OUTPUT_DIR,
                   n_bootstrap=N_BOOTSTRAP, force=RECOMPUTE)
display(manifest)

## Effects by level and original metric checks

Delta is the reconciled result minus base; negative values indicate lower error.
Compute MASE/RMSSE per node, then average equally within each semantic level. Compare the check table with aggregates of the current per-node metrics.

In [ ]:
for dataset in ['ausgrid2013','gef2017']:
    folder = OUTPUT_DIR / dataset
    print(dataset)
    display(pd.read_csv(folder / 'level_effects_bootstrap.csv'))
    checks = pd.read_csv(folder / 'reproduction_check.csv')
    display(checks.groupby('method')[['abs_diff_MASE','abs_diff_RMSSE']].max())
    assert checks[['abs_diff_MASE','abs_diff_RMSSE']].to_numpy().max() <= 1e-6

## Bottom-level node improvement rates and large-error observations

In [ ]:
for dataset in ['ausgrid2013','gef2017']:
    folder = OUTPUT_DIR / dataset
    print(dataset)
    display(pd.read_csv(folder / 'appendix_node_improvement.csv'))
    display(pd.read_csv(folder / 'largest_error_decile.csv'))
    display(Image(filename=str(folder / 'error_redistribution_heatmap.png')))

## Interpretation

Error deciles are defined by base errors, and patterns across groups are descriptive.
Bootstrap intervals describe changes within the saved experiments, not stability across training seeds.

In [ ]:
for dataset in ['ausgrid2013','gef2017']:
    records = json.loads((OUTPUT_DIR / dataset / 'validation.json').read_text())
    display(pd.DataFrame(records))
assert len(manifest) == 12
print('All outputs:', OUTPUT_DIR)
print('Source fingerprints:', OUTPUT_DIR / 'source_manifest.json')